In [1]:
import sys
from pathlib import Path

root = Path(r"D:\Projects\speech-ai-learning")

if root not in sys.path:
    sys.path.insert(0, str(root))

from src.ctc import extend_target, ctc_forward_log


# Test 1: 普通目标
assert extend_target([1, 2, 3]) == [
    0, 1, 0, 2, 0, 3, 0
]

# Test 2: 重复 Token
assert extend_target([1, 1]) == [
    0, 1, 0, 1, 0
]

# Test 3: 空目标
assert extend_target([]) == [0]

# Test 4: 自定义 Blank
assert extend_target([2, 3], blank=4) == [
    4, 2, 4, 3, 4
]

# Test 5: 长度检查
target = [1, 2, 3, 4]
extended = extend_target(target)

assert len(extended) == 2 * len(target) + 1

# Test 6: target 不允许包含 Blank
try:
    extend_target([1, 0, 2])
except ValueError:
    pass
else:
    raise AssertionError("Blank should not appear in target")

print("All extend_target tests passed!")


All extend_target tests passed!


In [2]:
#ctc-forward正确性测试
#测试1
import torch
probs = torch.tensor(
    [
        [0.6, 0.4],
        [0.3, 0.7]
    ],
    dtype=torch.float32
)

log_probs = torch.log(probs)

result = ctc_forward_log(
    log_probs,
    target=[1],
    blank=0
)

assert torch.allclose(
    result.exp(),
    torch.tensor(0.82)
)

print("Test 1 passed:", result.exp().item())

#测试2
probs = torch.tensor(
    [
        [0.5, 0.5],
        [0.5, 0.5]
    ],
    dtype=torch.float32
)

log_probs = torch.log(probs)

result = ctc_forward_log(
    log_probs,
    target=[1, 1],
    blank=0
)

assert torch.isneginf(result)

print("Test 2 passed")

#测试3
probs = torch.tensor(
    [
        [0.6, 0.4],
        [0.3, 0.7]
    ],
    dtype=torch.float32
)

log_probs = torch.log(probs)

result = ctc_forward_log(
    log_probs,
    target=[],
    blank=0
)

expected = torch.log(
    torch.tensor(0.6 * 0.3)
)

assert torch.allclose(
    result,
    expected
)

print("Test 3 passed")

#测试4
log_probs = torch.empty(
    (0, 2),
    dtype=torch.float32
)

# 空目标
result_empty = ctc_forward_log(
    log_probs,
    target=[],
    blank=0
)

assert result_empty.item() == 0.0

# 非空目标
result_nonempty = ctc_forward_log(
    log_probs,
    target=[1],
    blank=0
)

assert torch.isneginf(result_nonempty)

print("Test 4 passed")

Test 1 passed: 0.8200000524520874
Test 2 passed
Test 3 passed
Test 4 passed
